# Extraction Quality Evaluation

Evaluates the LLM extractor against CUAD gold-standard labels.  
Metrics: Precision, Recall, F1 per clause type.

**Dataset:** 31 CUAD contracts · 3 clause types (Price, Penalty, Renewal)  
**Mode:** Results below use `USE_MOCK_EXTRACTOR=True`; flip to `False` for real scores.

In [ ]:
import sys, os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Resolve project root whether notebook is run from project root or notebooks/
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import config

conn = duckdb.connect(str(config.DB_PATH), read_only=True)

price_df   = conn.execute('SELECT * FROM price_clauses').df()
penalty_df = conn.execute('SELECT * FROM penalty_clauses').df()
renewal_df = conn.execute('SELECT * FROM renewal_clauses').df()

print(f'price_clauses   : {price_df.shape}')
print(f'penalty_clauses : {penalty_df.shape}')
print(f'renewal_clauses : {renewal_df.shape}')
print(f'\nDB path: {config.DB_PATH}')
print(f'Mock mode: {config.USE_MOCK_EXTRACTOR}')

## Mock Mode Baseline

With `USE_MOCK_EXTRACTOR = True` in `config.py`, every contract is assigned
clauses from one of three hardcoded templates (cycled by `hash(contract_id) % 3`).

**Expected behaviour:**
- Every contract receives exactly **1 clause of each type** (Price, Penalty, Renewal).
- Vendor names are always one of: *Acme Supplies LLC*, *Global Logistics Partners*, *TechPro Solutions Inc.*
- Confidence scores are fixed at **0.85–0.99** depending on the template.

This section documents that baseline and sets up the evaluation framework so it runs
automatically once live extraction is enabled.

> **Flip to live mode:** set `USE_MOCK_EXTRACTOR = False` in `config.py`,
> add your `ANTHROPIC_API_KEY` to `.env`, re-run `extraction/run_extraction.py`,
> then re-execute this notebook to obtain real precision / recall scores.

In [ ]:
# ── Clause count bar chart ────────────────────────────────────────────────────
types  = ['Price', 'Penalty', 'Renewal']
counts = [len(price_df), len(penalty_df), len(renewal_df)]
colors = ['#3B82F6', '#EF4444', '#10B981']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(types, counts, color=colors)
for bar, count in zip(axes[0].patches, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(count), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Extracted Clause Counts by Type')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, max(counts) * 1.2)

# ── Confidence distribution ───────────────────────────────────────────────────
for df, label, color in zip([price_df, penalty_df, renewal_df], types, colors):
    if not df.empty:
        axes[1].hist(df['confidence'], bins=10, alpha=0.6, label=label, color=color)
axes[1].set_title('Confidence Score Distribution')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'clause_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebooks/clause_distribution.png')

# ── Unique vendors per clause type ───────────────────────────────────────────
print('\nUnique vendors per clause type:')
for label, df in zip(types, [price_df, penalty_df, renewal_df]):
    print(f'  {label:<8}: {sorted(df["vendor_name"].unique().tolist())}')

# ── Confidence stats ─────────────────────────────────────────────────────────
print('\nConfidence score statistics:')
conf_rows = []
for label, df in zip(types, [price_df, penalty_df, renewal_df]):
    if not df.empty:
        conf_rows.append({
            'Clause Type': label,
            'Min':  round(df['confidence'].min(),  3),
            'Mean': round(df['confidence'].mean(), 3),
            'Max':  round(df['confidence'].max(),  3),
            'Count': len(df),
        })
conf_stats = pd.DataFrame(conf_rows)
print(conf_stats.to_string(index=False))

## CUAD Label Alignment

The [CUAD dataset](https://www.atticusprojectai.org/cuad) provides expert-labelled spans
for 41 contract clause categories. Three map directly to our extraction schema:

| CUAD Category | Our Schema | Notes |
|---|---|---|
| **Price Restrictions** | `PriceClause` | Caps, MFN clauses, unit pricing |
| **Liquidated Damages** | `PenaltyClause` | Fixed / per-diem / percentage penalties |
| **Renewal Term** | `RenewalClause` | Auto-renewal, notice periods, expiry |

### Precision / Recall methodology

Because our extractor works at **contract level** (not span level), we use a
binary contract-level metric:

- A contract is a **True Positive (TP)** if at least one clause of the correct type
  was extracted from it.
- A contract is a **False Negative (FN)** if the CUAD label is present but we
  extracted zero clauses of that type.
- A contract is a **False Positive (FP)** if we extracted a clause but CUAD has
  no matching label.

```
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1        = 2 * Precision * Recall / (Precision + Recall)
```

> **In mock mode** FP/FN cannot be computed (no real extraction).  
> The recall proxy below uses `extracted / total_contracts` as a coverage metric.

In [ ]:
from IPython.display import display

TOTAL_CONTRACTS = conn.execute('SELECT COUNT(*) FROM contracts').fetchone()[0]

# Contracts with at least one clause of each type
price_covered   = price_df['contract_id'].nunique()
penalty_covered = penalty_df['contract_id'].nunique()
renewal_covered = renewal_df['contract_id'].nunique()

eval_df = pd.DataFrame({
    'Clause Type':      ['Price', 'Penalty', 'Renewal'],
    'Contracts w/ Clause': [price_covered, penalty_covered, renewal_covered],
    'Total Contracts':  [TOTAL_CONTRACTS] * 3,
    'Recall Proxy':     [
        price_covered   / TOTAL_CONTRACTS,
        penalty_covered / TOTAL_CONTRACTS,
        renewal_covered / TOTAL_CONTRACTS,
    ],
    'Note': [
        'Mock: 1 clause per contract by design',
        'Mock: 1 clause per contract by design',
        'Mock: 1 clause per contract by design',
    ],
})

print(f'Total contracts in DB : {TOTAL_CONTRACTS}')
print()
print('NOTE: In mock mode recall = 1.0 by design — every contract is assigned')
print('      one clause of each type from the hardcoded template pool.')
print('      Precision requires gold labels; compute after live extraction.')
print()

styled = (
    eval_df
    .style
    .format({'Recall Proxy': '{:.1%}'})
    .background_gradient(subset=['Recall Proxy'], cmap='RdYlGn', vmin=0, vmax=1)
    .set_caption('Contract-level Recall Proxy (Mock Mode)')
)
display(styled)

## Compliance Summary

The gap engine (`compliance/gap_engine.py`) joins extracted clause data against
SCMS shipment PO actuals to surface three categories of compliance risk.

**Current findings (mock extraction):**

- **Price Breaches:** 0 detected. Mock vendor names (*Acme Supplies LLC* etc.) do not
  match SCMS pharmaceutical supplier names, so no join keys overlap. With live extraction,
  vendor names will be drawn from actual contract text and matches will occur.

- **Penalty Exposure:** 0 detected. Same join-key issue as price breaches.

- **Renewal Alerts:** **31 contracts flagged as expired.** All mock renewal clauses
  have hardcoded expiry dates in 2024–2026, all of which have already passed.
  This correctly exercises the alert pipeline end-to-end.

The renewal alert visualization below is therefore the most meaningful output
available in mock mode.

In [ ]:
import matplotlib.dates as mdates
from datetime import date

renewal_alerts = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'compliance_renewal_alerts.csv')
renewal_alerts['expiry_date'] = pd.to_datetime(renewal_alerts['expiry_date'])

print(f'Renewal alerts loaded: {len(renewal_alerts)} rows')
print(renewal_alerts['status'].value_counts().to_string())
print()

color_map = {'expired': '#EF4444', 'urgent': '#F97316', 'upcoming': '#EAB308'}
today = pd.Timestamp(date.today())

fig, ax = plt.subplots(figsize=(13, 7))

for status, grp in renewal_alerts.groupby('status'):
    ax.scatter(
        grp['expiry_date'],
        grp['vendor_name'],
        c=color_map.get(status, '#6B7280'),
        label=status.capitalize(),
        s=90, zorder=3, edgecolors='white', linewidths=0.5,
    )

# Today marker
ax.axvline(today, color='#1D4ED8', linestyle='--', linewidth=1.5,
           label=f'Today ({today.date()})')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_title('Contract Renewal Timeline', fontsize=14, fontweight='bold')
ax.set_xlabel('Expiry Date')
ax.set_ylabel('Vendor')
ax.legend(loc='upper left')
ax.grid(axis='x', linestyle='--', alpha=0.35)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'renewal_timeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebooks/renewal_timeline.png')

## Next Steps

To obtain **real precision / recall scores** against CUAD gold labels:

1. **Enable live extraction** — set `USE_MOCK_EXTRACTOR = False` in `config.py`
2. **Add your API key** — set `ANTHROPIC_API_KEY` in `.env`
3. **Re-run extraction** — `python extraction/run_extraction.py` (31 contracts × ~3 API calls each)
4. **Re-execute this notebook** — `jupyter nbconvert --to notebook --execute notebooks/eval_extraction.ipynb`
5. **Download CUAD annotation JSON** from Kaggle and add a cell that loads the gold labels,
   computes TP/FP/FN per contract per clause type, and outputs a full Precision/Recall/F1 table.

**Estimated live extraction cost:** ~$0.10–0.30 USD for all 31 contracts at current
Claude API pricing (depends on context length and chunk filtering).